# Orthopedic Joint Identification (Shoulder / Hip / Knee) from X-Ray Images
### Comparing a Classical ML baseline, a Custom CNN, and EfficientNet-B3 Transfer Learning

**Pipeline:** Data acquisition (Kaggle) → EDA → Preprocessing → Model 1 (Random Forest) → Model 2 (Custom CNN) → Model 3 (EfficientNet-B3) → Evaluation → Model Comparison

This notebook reproduces the local project end-to-end, but is written for **Google Colab with a free GPU runtime** (Runtime → Change runtime type → GPU), which removes the CPU bottleneck we hit locally and lets EfficientNet-B3 train at its native 300×300 resolution for the full epoch budget.

## 1. Setup & Imports

In [ ]:
!pip -q install kaggle scikit-image seaborn --upgrade

import os, time, json, random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

from skimage.feature import hog
from skimage.color import rgb2gray

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                              classification_report, confusion_matrix, f1_score)

import warnings
warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

## 2. Kaggle Authentication & Dataset Download

We combine **three real, separately published Kaggle datasets** into one Shoulder/Hip/Knee joint-identification dataset — no single dataset covers all three joints, and no synthetic data is used anywhere.

| Class | Kaggle dataset | License |
|---|---|---|
| Shoulder | `yasserhessein/shoulder-implant-xray` | Public |
| Hip | `tawsifurrahman/aseptic-loose-hip-implant-xray-database` | CC0-1.0 |
| Knee | `tommyngx/digital-knee-xray` | CC BY 4.0 |

**Two ways to authenticate — use whichever you have:**
- **New-style API token** (a string starting with `KGAT_...`, from Kaggle → Settings → API → Create New Token): paste it into `KAGGLE_TOKEN` in the next cell. No file needed.
- **Classic `kaggle.json`** (older Kaggle accounts): leave `KAGGLE_TOKEN` empty and run the cell — it will prompt you to upload the file instead.


In [ ]:
KAGGLE_TOKEN = ""  # <-- paste your KGAT_... token here (from Kaggle Settings > API), or leave empty to upload kaggle.json instead

os.makedirs("/root/.kaggle", exist_ok=True)

if KAGGLE_TOKEN.strip():
    # New-style token: kaggle CLI reads this from the environment, no file needed
    os.environ["KAGGLE_API_TOKEN"] = KAGGLE_TOKEN.strip()
    print("Using KAGGLE_API_TOKEN environment variable.")
else:
    from google.colab import files
    print("No token pasted above -- upload your kaggle.json API file instead:")
    uploaded = files.upload()
    !cp kaggle.json /root/.kaggle/kaggle.json
    !chmod 600 /root/.kaggle/kaggle.json
    print("kaggle.json installed to /root/.kaggle/kaggle.json")


In [ ]:
!mkdir -p /content/data/shoulder /content/data/hip /content/data/knee

!kaggle datasets download -d yasserhessein/shoulder-implant-xray -p /content/data/shoulder --unzip -q
!kaggle datasets download -d tawsifurrahman/aseptic-loose-hip-implant-xray-database -p /content/data/hip --unzip -q
!kaggle datasets download -d tommyngx/digital-knee-xray -p /content/data/knee --unzip -q

print("All three datasets downloaded.")

## 3. Build the Combined Joint-Identification Dataset

- **Shoulder**: 4 manufacturer subtypes (Cofield/Depuy/Tornier/Zimmer) pooled into one `Shoulder` label.
- **Hip**: `Control` + `Loose` subtypes pooled into one `Hip` label.
- **Knee**: 5 osteoarthritis-severity folders sampled evenly and pooled into one `Knee` label.
- Shoulder & Knee are capped to ~300 images each to reduce class imbalance against Hip's natural ~206 images.

In [ ]:
SHOULDER_DIR = Path("/content/data/shoulder")
HIP_DIR = Path("/content/data/hip")   # contains Data/Control, Data/Loose
KNEE_DIR = Path("/content/data/knee") # contains MedicalExpert-I/<grade folders>
PER_CLASS_CAP = 300

def list_images(d):
    exts = {".jpg", ".jpeg", ".png"}
    return [str(p) for p in Path(d).rglob("*") if p.suffix.lower() in exts]

rows = []

# Shoulder: pool all manufacturer subtypes
shoulder_paths = list_images(SHOULDER_DIR)
random.shuffle(shoulder_paths)
for p in shoulder_paths[:PER_CLASS_CAP]:
    rows.append({"path": p, "label": "Shoulder"})

# Hip: pool Control + Loose
hip_paths = list_images(HIP_DIR)
for p in hip_paths:
    rows.append({"path": p, "label": "Hip"})

# Knee: sample evenly across severity-grade folders
knee_root = next(Path(KNEE_DIR).glob("MedicalExpert-I"))
knee_grade_dirs = sorted([d for d in knee_root.iterdir() if d.is_dir()])
per_grade = max(1, PER_CLASS_CAP // len(knee_grade_dirs))
knee_paths = []
for d in knee_grade_dirs:
    imgs = list_images(d)
    random.shuffle(imgs)
    knee_paths.extend(imgs[:per_grade])
random.shuffle(knee_paths)
for p in knee_paths[:PER_CLASS_CAP]:
    rows.append({"path": p, "label": "Knee"})

df = pd.DataFrame(rows).drop_duplicates(subset="path").reset_index(drop=True)
print(df["label"].value_counts())
print("Total images:", len(df))

## 4. Exploratory Data Analysis

In [ ]:
# Class distribution
counts = df["label"].value_counts()
plt.figure(figsize=(6,4))
bars = plt.bar(counts.index, counts.values, color=["#4C72B0","#DD8452","#55A868"])
for b, v in zip(bars, counts.values):
    plt.text(b.get_x()+b.get_width()/2, v+3, str(v), ha="center")
plt.title(f"Class distribution (n={len(df)})")
plt.ylabel("Number of images")
plt.show()

print("Imbalance ratio (max/min):", round(counts.max()/counts.min(), 2))

In [ ]:
# Corrupt-image check + dimension stats
sizes, corrupt = [], []
for p in df["path"]:
    try:
        with Image.open(p) as im:
            im.verify()
        with Image.open(p) as im:
            sizes.append(im.size)
    except Exception as e:
        corrupt.append((p, str(e)))

widths = [s[0] for s in sizes]; heights = [s[1] for s in sizes]
print("Corrupt images:", len(corrupt))
print(f"Width:  min={min(widths)} max={max(widths)} mean={np.mean(widths):.0f}")
print(f"Height: min={min(heights)} max={max(heights)} mean={np.mean(heights):.0f}")

plt.figure(figsize=(5,5))
plt.scatter(widths, heights, alpha=0.3, s=12)
plt.xlabel("Width (px)"); plt.ylabel("Height (px)")
plt.title("Raw image dimensions across all 3 sources")
plt.show()

In [ ]:
# Sample grid, 6 images per class
fig, axes = plt.subplots(3, 6, figsize=(14,7))
for i, cls in enumerate(counts.index):
    sample_paths = df[df["label"]==cls]["path"].sample(6, random_state=SEED).tolist()
    for j, p in enumerate(sample_paths):
        axes[i, j].imshow(Image.open(p).convert("RGB"))
        axes[i, j].axis("off")
    axes[i, 0].text(-0.3, 0.5, cls, transform=axes[i,0].transAxes, fontsize=13,
                     va="center", ha="right", rotation=90)
plt.suptitle("Sample X-ray images per joint class")
plt.tight_layout()
plt.show()

## 5. Preprocessing: Train/Validation/Test Split

A single **stratified 70/15/15 split** (fixed seed) is reused identically across all three models so results are directly comparable.

In [ ]:
train_df, temp_df = train_test_split(df, test_size=0.30, stratify=df["label"], random_state=SEED)
val_df, test_df = train_test_split(temp_df, test_size=0.50, stratify=temp_df["label"], random_state=SEED)

print("train", len(train_df), train_df["label"].value_counts().to_dict())
print("val  ", len(val_df), val_df["label"].value_counts().to_dict())
print("test ", len(test_df), test_df["label"].value_counts().to_dict())

le = LabelEncoder()
le.fit(train_df["label"])
CLASS_NAMES = list(le.classes_)
print("Classes:", CLASS_NAMES)

## 6. Model 1 — Random Forest (Classical ML Baseline)

**Preprocessing for this model:** resize to 128×128 → grayscale → HOG descriptor (edge/contour structure) + 32-bin RGB color histograms (density/brightness) → concatenate → standardize.

**Why:** tests whether simple handcrafted features are already sufficient before reaching for a deep network, and gives a fast, interpretable, low-compute baseline.

In [ ]:
IMG_SIZE_RF = (128, 128)

def extract_features(path):
    img = Image.open(path).convert("RGB").resize(IMG_SIZE_RF)
    arr = np.array(img)
    gray = rgb2gray(arr)
    hog_feat = hog(gray, orientations=9, pixels_per_cell=(16,16), cells_per_block=(2,2),
                    block_norm="L2-Hys", feature_vector=True)
    hist_feat = np.concatenate([
        np.histogram(arr[:,:,c], bins=32, range=(0,255), density=True)[0] for c in range(3)
    ])
    return np.concatenate([hog_feat, hist_feat])

def build_feature_matrix(frame):
    return np.vstack([extract_features(p) for p in frame["path"]])

t0 = time.time()
X_train = build_feature_matrix(train_df); y_train = le.transform(train_df["label"])
X_val   = build_feature_matrix(val_df);   y_val   = le.transform(val_df["label"])
X_test  = build_feature_matrix(test_df);  y_test  = le.transform(test_df["label"])
print(f"Feature extraction done in {time.time()-t0:.1f}s, dim={X_train.shape[1]}")

In [ ]:
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("rf", RandomForestClassifier(class_weight="balanced", random_state=SEED)),
])
param_grid = {
    "rf__n_estimators": [200, 400],
    "rf__max_depth": [None, 20],
    "rf__min_samples_leaf": [1, 3],
}
grid = GridSearchCV(pipe, param_grid, cv=3, scoring="f1_macro", n_jobs=-1)

t0 = time.time()
grid.fit(X_train, y_train)
rf_train_time = time.time() - t0
rf_model = grid.best_estimator_
print("Best params:", grid.best_params_, " CV f1_macro=%.3f" % grid.best_score_)

In [ ]:
t0 = time.time()
rf_test_pred = rf_model.predict(X_test)
rf_infer_ms = (time.time()-t0)/len(y_test)*1000

rf_acc = accuracy_score(y_test, rf_test_pred)
rf_prec, rf_rec, rf_f1, _ = precision_recall_fscore_support(y_test, rf_test_pred, average="macro", zero_division=0)
rf_report = classification_report(y_test, rf_test_pred, target_names=CLASS_NAMES, output_dict=True, zero_division=0)
rf_cm = confusion_matrix(y_test, rf_test_pred)

print(f"Model 1 (Random Forest) — Accuracy: {rf_acc:.4f}  F1(macro): {rf_f1:.4f}")

plt.figure(figsize=(5,4.3))
sns.heatmap(rf_cm, annot=True, fmt="d", cmap="Blues", xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.title("Model 1: Random Forest — Confusion Matrix")
plt.show()

## 7. Shared PyTorch Dataset & Training Utilities

Used by both Model 2 (custom CNN) and Model 3 (EfficientNet-B3).

**Preprocessing for these models:** resize → train-only augmentation (horizontal flip, ±10° rotation, brightness/contrast jitter) to fight overfitting on a small dataset → normalize → class-weighted loss to counter the residual class imbalance.

In [ ]:
class JointXrayDataset(Dataset):
    def __init__(self, frame, label_encoder, transform=None):
        self.paths = frame["path"].tolist()
        self.labels = label_encoder.transform(frame["label"])
        self.transform = transform
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, int(self.labels[idx])

def get_transforms(mean, std, size, augment=False):
    ops = [transforms.Resize((size, size))]
    if augment:
        ops += [
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomRotation(10),
            transforms.ColorJitter(brightness=0.15, contrast=0.15),
        ]
    ops += [transforms.ToTensor(), transforms.Normalize(mean, std)]
    return transforms.Compose(ops)

# Full-GPU-utilization settings: cuDNN autotuner + automatic mixed precision (fp16 on GPU tensor cores)
torch.backends.cudnn.benchmark = True
USE_AMP = (DEVICE.type == "cuda")
print("Mixed precision (AMP) enabled:", USE_AMP)

def run_epoch(model, loader, criterion, optimizer, device, train=True, scaler=None):
    model.train() if train else model.eval()
    total_loss, all_preds, all_labels = 0.0, [], []
    torch.set_grad_enabled(train)
    for x, y in loader:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        if train:
            optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type=device.type, enabled=USE_AMP):
            out = model(x)
            loss = criterion(out, y)
        if train:
            if scaler is not None:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            else:
                loss.backward()
                optimizer.step()
        total_loss += loss.item() * x.size(0)
        all_preds.extend(out.argmax(1).detach().cpu().numpy().tolist())
        all_labels.extend(y.detach().cpu().numpy().tolist())
    torch.set_grad_enabled(True)
    return total_loss/len(loader.dataset), accuracy_score(all_labels, all_preds), all_preds, all_labels

def evaluate_full(model, loader, device, class_names):
    model.eval()
    all_preds, all_labels = [], []
    t0 = time.time()
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device, non_blocking=True)
            with torch.autocast(device_type=device.type, enabled=USE_AMP):
                out = model(x)
            all_preds.extend(out.argmax(1).cpu().numpy().tolist())
            all_labels.extend(y.numpy().tolist())
    infer_ms = (time.time()-t0)/len(all_labels)*1000
    acc = accuracy_score(all_labels, all_preds)
    prec, rec, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average="macro", zero_division=0)
    report = classification_report(all_labels, all_preds, target_names=class_names, output_dict=True, zero_division=0)
    cm = confusion_matrix(all_labels, all_preds)
    return {"accuracy": acc, "precision_macro": prec, "recall_macro": rec, "f1_macro": f1,
            "avg_inference_time_ms": infer_ms}, report, cm


## 8. Model 2 — Custom CNN (Trained From Scratch)

A compact 4-block convolutional network with no pretraining, to see how much a lightweight CNN can learn purely from ~560 training images.

In [ ]:
MEAN_CNN, STD_CNN = [0.5,0.5,0.5], [0.5,0.5,0.5]
IMG_SIZE_CNN = 128
EPOCHS_CNN = 25
BATCH = 16

class SmallCNN(nn.Module):
    def __init__(self, num_classes=3):
        super().__init__()
        def block(cin, cout):
            return nn.Sequential(
                nn.Conv2d(cin, cout, 3, padding=1), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
                nn.Conv2d(cout, cout, 3, padding=1), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
                nn.MaxPool2d(2),
            )
        self.features = nn.Sequential(block(3,32), block(32,64), block(64,128), block(128,256))
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Dropout(0.4),
            nn.Linear(256,128), nn.ReLU(inplace=True), nn.Dropout(0.3), nn.Linear(128, num_classes),
        )
    def forward(self, x):
        return self.classifier(self.features(x))

train_ds_cnn = JointXrayDataset(train_df, le, get_transforms(MEAN_CNN, STD_CNN, IMG_SIZE_CNN, augment=True))
val_ds_cnn   = JointXrayDataset(val_df,   le, get_transforms(MEAN_CNN, STD_CNN, IMG_SIZE_CNN, augment=False))
test_ds_cnn  = JointXrayDataset(test_df,  le, get_transforms(MEAN_CNN, STD_CNN, IMG_SIZE_CNN, augment=False))

train_loader_cnn = DataLoader(train_ds_cnn, batch_size=BATCH, shuffle=True)
val_loader_cnn   = DataLoader(val_ds_cnn,   batch_size=BATCH, shuffle=False)
test_loader_cnn  = DataLoader(test_ds_cnn,  batch_size=BATCH, shuffle=False)

class_counts = train_df["label"].value_counts().reindex(CLASS_NAMES)
weights = (1.0/class_counts).values
weights = weights/weights.sum()*len(CLASS_NAMES)
class_weights_cnn = torch.tensor(weights, dtype=torch.float32).to(DEVICE)

cnn_model = SmallCNN(num_classes=len(CLASS_NAMES)).to(DEVICE)
cnn_params = sum(p.numel() for p in cnn_model.parameters())
print("Custom CNN parameters:", cnn_params)

In [ ]:
criterion = nn.CrossEntropyLoss(weight=class_weights_cnn)
optimizer = torch.optim.Adam(cnn_model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=3)

history_cnn = {"train_loss":[], "val_loss":[], "train_acc":[], "val_acc":[]}
best_val_f1, best_state = -1, None
t0 = time.time()
for epoch in range(1, EPOCHS_CNN+1):
    tr_loss, tr_acc, _, _ = run_epoch(cnn_model, train_loader_cnn, criterion, optimizer, DEVICE, train=True)
    val_loss, val_acc, val_preds, val_labels = run_epoch(cnn_model, val_loader_cnn, criterion, optimizer, DEVICE, train=False)
    scheduler.step(val_loss)
    val_f1 = f1_score(val_labels, val_preds, average="macro", zero_division=0)
    history_cnn["train_loss"].append(tr_loss); history_cnn["val_loss"].append(val_loss)
    history_cnn["train_acc"].append(tr_acc);   history_cnn["val_acc"].append(val_acc)
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_state = {k: v.clone() for k, v in cnn_model.state_dict().items()}
    print(f"Epoch {epoch:02d}/{EPOCHS_CNN} | train_loss={tr_loss:.4f} acc={tr_acc:.3f} | val_loss={val_loss:.4f} acc={val_acc:.3f} f1={val_f1:.3f}")
cnn_train_time = time.time() - t0
cnn_model.load_state_dict(best_state)

In [ ]:
cnn_test_metrics, cnn_report, cnn_cm = evaluate_full(cnn_model, test_loader_cnn, DEVICE, CLASS_NAMES)
print("Model 2 (Custom CNN) —", cnn_test_metrics)

fig, ax = plt.subplots(1, 2, figsize=(10,4))
ax[0].plot(history_cnn["train_loss"], label="train"); ax[0].plot(history_cnn["val_loss"], label="val")
ax[0].set_title("Model 2: CNN — Loss"); ax[0].set_xlabel("Epoch"); ax[0].legend()
ax[1].plot(history_cnn["train_acc"], label="train"); ax[1].plot(history_cnn["val_acc"], label="val")
ax[1].set_title("Model 2: CNN — Accuracy"); ax[1].set_xlabel("Epoch"); ax[1].legend()
plt.tight_layout(); plt.show()

plt.figure(figsize=(5,4.3))
sns.heatmap(cnn_cm, annot=True, fmt="d", cmap="Greens", xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.title("Model 2: Custom CNN — Confusion Matrix")
plt.show()

## 9. Model 3 — EfficientNet-B3 (ImageNet Transfer Learning, Flagship Model)

Pretrained on ImageNet-1K; classifier head replaced for 3-class output. Differential learning rates (small for the pretrained backbone, larger for the new head), cosine LR schedule, label smoothing, class-weighted loss.

On Colab's GPU this runs at the **native 300×300 resolution** for the full epoch budget (unlike the CPU-constrained local run, which had to drop to 224×224 / 6 epochs).

In [ ]:
IMG_SIZE_EFF = 300
EPOCHS_EFF = 20
LR_HEAD, LR_BACKBONE = 1e-3, 1e-4
IMAGENET_MEAN, IMAGENET_STD = [0.485,0.456,0.406], [0.229,0.224,0.225]
BATCH_EFF = 32  # larger batch -- T4 (16GB) comfortably fits EfficientNet-B3 at 300x300 with batch 32

weights = models.EfficientNet_B3_Weights.IMAGENET1K_V1
eff_model = models.efficientnet_b3(weights=weights)
in_feat = eff_model.classifier[1].in_features
eff_model.classifier = nn.Sequential(nn.Dropout(0.4), nn.Linear(in_feat, len(CLASS_NAMES)))
eff_model = eff_model.to(DEVICE)
eff_model = eff_model.to(memory_format=torch.channels_last) if DEVICE.type == "cuda" else eff_model

eff_params = sum(p.numel() for p in eff_model.parameters())
print("EfficientNet-B3 parameters:", eff_params)

NUM_WORKERS = os.cpu_count()  # fully utilize all available CPU cores for data loading/augmentation
print("DataLoader workers (CPU cores):", NUM_WORKERS)

train_ds_eff = JointXrayDataset(train_df, le, get_transforms(IMAGENET_MEAN, IMAGENET_STD, IMG_SIZE_EFF, augment=True))
val_ds_eff   = JointXrayDataset(val_df,   le, get_transforms(IMAGENET_MEAN, IMAGENET_STD, IMG_SIZE_EFF, augment=False))
test_ds_eff  = JointXrayDataset(test_df,  le, get_transforms(IMAGENET_MEAN, IMAGENET_STD, IMG_SIZE_EFF, augment=False))

loader_kwargs = dict(num_workers=NUM_WORKERS, pin_memory=(DEVICE.type=="cuda"), persistent_workers=(NUM_WORKERS>0))
train_loader_eff = DataLoader(train_ds_eff, batch_size=BATCH_EFF, shuffle=True, **loader_kwargs)
val_loader_eff   = DataLoader(val_ds_eff,   batch_size=BATCH_EFF, shuffle=False, **loader_kwargs)
test_loader_eff  = DataLoader(test_ds_eff,  batch_size=BATCH_EFF, shuffle=False, **loader_kwargs)


In [ ]:
# reuse the same inverse-frequency class weights computed for Model 2
criterion_eff = nn.CrossEntropyLoss(weight=class_weights_cnn, label_smoothing=0.05)

head_ids = set(id(p) for p in eff_model.classifier.parameters())
backbone_params = [p for p in eff_model.parameters() if id(p) not in head_ids]
head_params = list(eff_model.classifier.parameters())
optimizer_eff = torch.optim.AdamW([
    {"params": backbone_params, "lr": LR_BACKBONE},
    {"params": head_params, "lr": LR_HEAD},
], weight_decay=1e-4)
scheduler_eff = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_eff, T_max=EPOCHS_EFF)
scaler_eff = torch.amp.GradScaler(enabled=USE_AMP)  # scales gradients for fp16 mixed-precision training

history_eff = {"train_loss":[], "val_loss":[], "train_acc":[], "val_acc":[], "val_f1":[]}
best_val_f1_eff, best_state_eff, patience, bad_epochs = -1, None, 4, 0
t0 = time.time()
for epoch in range(1, EPOCHS_EFF+1):
    tr_loss, tr_acc, _, _ = run_epoch(eff_model, train_loader_eff, criterion_eff, optimizer_eff, DEVICE, train=True, scaler=scaler_eff)
    val_loss, val_acc, val_preds, val_labels = run_epoch(eff_model, val_loader_eff, criterion_eff, optimizer_eff, DEVICE, train=False)
    scheduler_eff.step()
    val_f1 = f1_score(val_labels, val_preds, average="macro", zero_division=0)
    history_eff["train_loss"].append(tr_loss); history_eff["val_loss"].append(val_loss)
    history_eff["train_acc"].append(tr_acc);   history_eff["val_acc"].append(val_acc); history_eff["val_f1"].append(val_f1)
    if DEVICE.type == "cuda":
        mem_gb = torch.cuda.max_memory_allocated() / 1e9
        print(f"Epoch {epoch:02d}/{EPOCHS_EFF} | train_loss={tr_loss:.4f} acc={tr_acc:.3f} | val_loss={val_loss:.4f} acc={val_acc:.3f} f1={val_f1:.3f} | GPU peak mem={mem_gb:.2f}GB")
    else:
        print(f"Epoch {epoch:02d}/{EPOCHS_EFF} | train_loss={tr_loss:.4f} acc={tr_acc:.3f} | val_loss={val_loss:.4f} acc={val_acc:.3f} f1={val_f1:.3f}")
    if val_f1 > best_val_f1_eff:
        best_val_f1_eff = val_f1
        best_state_eff = {k: v.clone() for k, v in eff_model.state_dict().items()}
        bad_epochs = 0
    else:
        bad_epochs += 1
        if bad_epochs >= patience:
            print("Early stopping."); break
eff_train_time = time.time() - t0
eff_model.load_state_dict(best_state_eff)
print(f"
Total EfficientNet-B3 training time: {eff_train_time:.1f}s")


In [ ]:
eff_test_metrics, eff_report, eff_cm = evaluate_full(eff_model, test_loader_eff, DEVICE, CLASS_NAMES)
print("Model 3 (EfficientNet-B3) —", eff_test_metrics)

fig, ax = plt.subplots(1, 2, figsize=(10,4))
ax[0].plot(history_eff["train_loss"], label="train"); ax[0].plot(history_eff["val_loss"], label="val")
ax[0].set_title("Model 3: EfficientNet-B3 — Loss"); ax[0].set_xlabel("Epoch"); ax[0].legend()
ax[1].plot(history_eff["train_acc"], label="train"); ax[1].plot(history_eff["val_acc"], label="val")
ax[1].set_title("Model 3: EfficientNet-B3 — Accuracy"); ax[1].set_xlabel("Epoch"); ax[1].legend()
plt.tight_layout(); plt.show()

plt.figure(figsize=(5,4.3))
sns.heatmap(eff_cm, annot=True, fmt="d", cmap="Purples", xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.title("Model 3: EfficientNet-B3 — Confusion Matrix")
plt.show()

## 10. Model Comparison

In [ ]:
comparison = pd.DataFrame([
    {"Model": "Model 1: Random Forest (HOG features)",
     "Accuracy": rf_acc, "Precision": rf_prec, "Recall": rf_rec, "F1 (macro)": rf_f1,
     "Inference/img (ms)": rf_infer_ms, "Training time (s)": rf_train_time, "Params": "400 trees"},
    {"Model": "Model 2: Custom CNN (from scratch)",
     "Accuracy": cnn_test_metrics["accuracy"], "Precision": cnn_test_metrics["precision_macro"],
     "Recall": cnn_test_metrics["recall_macro"], "F1 (macro)": cnn_test_metrics["f1_macro"],
     "Inference/img (ms)": cnn_test_metrics["avg_inference_time_ms"], "Training time (s)": cnn_train_time,
     "Params": cnn_params},
    {"Model": "Model 3: EfficientNet-B3 (transfer learning)",
     "Accuracy": eff_test_metrics["accuracy"], "Precision": eff_test_metrics["precision_macro"],
     "Recall": eff_test_metrics["recall_macro"], "F1 (macro)": eff_test_metrics["f1_macro"],
     "Inference/img (ms)": eff_test_metrics["avg_inference_time_ms"], "Training time (s)": eff_train_time,
     "Params": eff_params},
])
for col in ["Accuracy","Precision","Recall","F1 (macro)"]:
    comparison[col] = (comparison[col]*100).round(2)
comparison

In [ ]:
fig, ax = plt.subplots(figsize=(9,5))
x = np.arange(len(comparison)); width = 0.35
ax.bar(x-width/2, comparison["Accuracy"], width, label="Accuracy")
ax.bar(x+width/2, comparison["F1 (macro)"], width, label="F1 (macro)")
ax.set_xticks(x); ax.set_xticklabels(["Model 1\nRF", "Model 2\nCNN", "Model 3\nEfficientNet-B3"])
ax.set_ylabel("Score (%)"); ax.set_ylim(0, 105)
ax.set_title("Model comparison — Shoulder/Hip/Knee joint identification")
ax.legend()
plt.tight_layout()
plt.show()

best_idx = comparison["F1 (macro)"].idxmax()
print(f"\nBest model: {comparison.loc[best_idx, 'Model']}  "
      f"(Accuracy={comparison.loc[best_idx,'Accuracy']}%, F1={comparison.loc[best_idx,'F1 (macro)']}%)")

## 11. Discussion & Conclusion

- **Joint identification is a visually easy task** — Shoulder, Hip, and Knee X-rays differ enormously in bone geometry and field of view, so even the classical Random Forest baseline (handcrafted HOG + color-histogram features) achieves very high accuracy with no learned representation at all.
- **EfficientNet-B3** (ImageNet-pretrained transfer learning) is expected to edge out the other two on macro-F1 thanks to its pretrained low-level filters and compound-scaled architecture, matching or exceeding the reference Orthonet benchmark (97.78% accuracy / 97.75% F1 on a related shoulder-implant task).
- **The custom from-scratch CNN** typically underperforms both — with only ~560 training images and no pretraining, it has the least reliable feature learning of the three, visible as more erratic validation-loss behavior across epochs.
- **Practical takeaway:** for an *easy* multi-class problem like this, a cheap classical baseline can be nearly as accurate as a 10M-parameter pretrained network — model complexity should be matched to task difficulty, not defaulted to the largest available architecture.

**Limitations:** small dataset (~800 images from 3 different source studies/hospitals, so cross-source protocol differences could act as a shortcut signal); no independent clinical validation of labels; task is joint identification only, not implant-manufacturer or abnormality detection.

**Future work:** extend to a second stage that identifies implant manufacturer/type once the joint is known; add Grad-CAM/saliency visualization for interpretability; train with a larger, multi-hospital dataset per joint.